# Colab Drive 工作流诊断入口

该 Notebook 用于 Colab 冷启动后的 Drive 持久化诊断: 挂载 Google Drive, 拉取仓库代码, 将本地 outputs 中的受治理产物镜像到 Drive, 写入工作流清单, 并在同一入口中执行一次清单重载校验。

运行依赖: 不依赖任何前序结果包, 不需要 GPU, 可在主方法或 baseline 正式运行前独立执行。该入口不参与任何论文运行层级的正式统计。

正式逻辑位于 `paper_workflow/colab_utils/drive_workflow.py`, Notebook 只作为远程执行入口。独立的 reload 入口已并入本 Notebook, 避免两个 Colab 入口维护同一套挂载、拉取和路径配置。

## 运行前准备

1. 确认 Google Drive 可挂载。
2. 默认 Drive 根目录为 `/content/drive/MyDrive/SLM`。
3. 该 Notebook 不加载真实模型, 只验证 Drive 持久化、工作流清单写入与清单重载。

In [ ]:
SLM_WM_PAPER_RUN_NAME = "probe_paper"

import os
os.environ["SLM_WM_PAPER_RUN_NAME"] = SLM_WM_PAPER_RUN_NAME

from google.colab import drive

drive.mount('/content/drive')
os.environ.setdefault('SLM_WM_DRIVE_ROOT', '/content/drive/MyDrive/SLM')
os.environ.setdefault('SLM_WM_WORKFLOW_NAME', 'colab_drive_workflow')
print({
    'drive_root': os.environ['SLM_WM_DRIVE_ROOT'],
    'workflow_name': os.environ['SLM_WM_WORKFLOW_NAME'],
})


In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

repository_url = os.environ.get("SLM_WM_REPOSITORY_URL", "https://github.com/RICHAAARC/SLM-WM.git")
repository_commit = os.environ.get("SLM_WM_REPOSITORY_COMMIT", "")
if re.fullmatch(r"[0-9a-f]{40}", repository_commit) is None:
    raise ValueError("正式运行前必须通过 SLM_WM_REPOSITORY_COMMIT 提供精确40位小写 Git SHA")
workspace_dir = Path(os.environ.get("SLM_WM_WORKSPACE_DIR", "/content/slm_wm_repository"))
workspace_dir.parent.mkdir(parents=True, exist_ok=True)
if workspace_dir.exists() and not (workspace_dir / ".git").is_dir():
    raise RuntimeError("SLM_WM_WORKSPACE_DIR 已存在但不是 Git 仓库")
if not (workspace_dir / ".git").is_dir():
    subprocess.run(["git", "clone", repository_url, str(workspace_dir)], check=True)

status_before_checkout = subprocess.run(
    ["git", "status", "--porcelain=v1", "--untracked-files=all"],
    cwd=workspace_dir,
    check=True,
    capture_output=True,
    text=True,
).stdout
if status_before_checkout:
    raise RuntimeError("复用 Colab 工作区前必须先清理 Git 工作树")
subprocess.run(["git", "fetch", "origin", "--tags", "--prune"], cwd=workspace_dir, check=True)
subprocess.run(["git", "rev-parse", "--verify", repository_commit + "^{commit}"], cwd=workspace_dir, check=True)
subprocess.run(["git", "checkout", "--detach", repository_commit], cwd=workspace_dir, check=True)

os.chdir(workspace_dir)
if str(workspace_dir) not in sys.path:
    sys.path.insert(0, str(workspace_dir))
from paper_workflow.colab_utils.formal_execution import verify_and_publish_formal_execution

formal_execution_lock = verify_and_publish_formal_execution(workspace_dir, repository_commit)
print({"workspace_dir": str(workspace_dir), "formal_execution_lock": formal_execution_lock})

In [ ]:
import os

from paper_workflow.colab_utils.drive_workflow import run_colab_drive_workflow

summary = run_colab_drive_workflow(
    root='.',
    drive_root=os.environ.get('SLM_WM_DRIVE_ROOT', '/content/drive/MyDrive/SLM'),
    workflow_name=os.environ.get('SLM_WM_WORKFLOW_NAME', 'colab_drive_workflow'),
    perform_mount=False,
)
summary


In [ ]:
from pathlib import Path

output_dir = Path('outputs/colab_drive_workflow')
for report_path in sorted(output_dir.glob('*.json')):
    print(report_path)
    print(report_path.read_text(encoding='utf-8')[:1200])

reload_path = output_dir / 'reload_smoke_record.jsonl'
if reload_path.exists():
    print(reload_path)
    print(reload_path.read_text(encoding='utf-8'))
